In [2]:
!pip install aiohttp nest_asyncio sentence-transformers

In [3]:
import os
import io
import asyncio
import aiohttp
import nest_asyncio
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm
import torch
from transformers import CLIPProcessor, CLIPModel

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
goods_path = '/content/drive/MyDrive/SmartSearch/lem_products.jsonl'
goods = pd.read_json(goods_path, lines=True)

goods.head()

,name,price,url,image_url
0,Триммер аккумуляторный GreenWorks 2107207SA 24...,5490,https://lemanapro.ru/product/trimmer-akkumulya...,https://cdn.lemanapro.ru/lmru/image/upload/dpr...
1,Триммер аккумуляторный GreenWorks G40LTK2 40 В...,11968,https://lemanapro.ru/product/trimmer-akkumulya...,https://cdn.lemanapro.ru/lmru/image/upload/dpr...
2,Газонокосилка аккумуляторная бесщеточная Green...,29990,https://lemanapro.ru/product/gazonokosilka-akk...,https://cdn.lemanapro.ru/lmru/image/upload/dpr...
3,Мотокоса бензиновая Oasis GT TB-2500N 3.3 л.с.,6990,https://lemanapro.ru/product/motokosa-benzinov...,https://cdn.lemanapro.ru/lmru/image/upload/dpr...
4,Газонокосилка электрическая Makita ELM3720 140...,13604,https://lemanapro.ru/product/gazonokosilka-ele...,https://cdn.lemanapro.ru/lmru/image/upload/dpr...


In [6]:
goods.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395846 entries, 0 to 395845
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   name       395846 non-null  object
 1   price      395846 non-null  int64 
 2   url        395846 non-null  object
 3   image_url  395846 non-null  object
dtypes: int64(1), object(3)
memory usage: 12.1+ MB


In [7]:
goods = goods.drop_duplicates(subset=['url']).reset_index(drop=True)
goods = goods.dropna(subset=['image_url'])

print(f"Уникальных товаров для обработки: {len(goods)}")

Уникальных товаров для обработки: 203734


In [8]:
# Разрешаем asyncio работать внутри ячеек Colab
nest_asyncio.apply()

# 1. Настройки
BATCH_SIZE = 64          # Качаем по 64 картинки за раз
SAVE_EVERY = 5000        # Частота сохранения чекпоинтов
EMB_DIR = '/content/drive/MyDrive/SmartSearch/embeddings/' # Папка на диске
os.makedirs(EMB_DIR, exist_ok=True)

# 2. Инициализация оригинального CLIP для картинок
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используем устройство: {device}")

model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to(device)

# 3. Асинхронная функция скачивания одной картинки
async def fetch_image(session, url, idx):
    try:
        # Небольшой таймаут, чтобы не зависать на битых ссылках
        async with session.get(url, timeout=10) as response:
            if response.status == 200:
                data = await response.read()
                img = Image.open(io.BytesIO(data)).convert("RGB")
                return idx, img
    except Exception:
        pass
    return idx, None

# 4. Функция обработки батча
async def process_batch(urls, start_idx):
    # Ограничиваем количество одновременных соединений, чтобы сервер не заблокировал
    connector = aiohttp.TCPConnector(limit=50)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [fetch_image(session, url, i) for i, url in enumerate(urls)]
        results = await asyncio.gather(*tasks)

    valid_images = []
    valid_indices = []

    for idx, img in results:
        if img is not None:
            valid_images.append(img)
            valid_indices.append(start_idx + idx)

    return valid_indices, valid_images

# 5. Главный конвейер
def run_pipeline(df):
    total_items = len(df)
    urls = df['image_url'].tolist()

    current_embeddings = []
    current_indices = []

    # Поиск чекпоинтов для продолжения работы
    existing_files = [f for f in os.listdir(EMB_DIR) if f.startswith('idx_') and f.endswith('.npy')]
    if existing_files:
        last_saved = max([int(f.split('_')[1].split('.')[0]) for f in existing_files])
        start_index = last_saved
        print(f"Найдено сохранение. Продолжаем с индекса {start_index}...")
    else:
        start_index = 0
        print("Начинаем обработку с нуля...")

    for i in tqdm(range(start_index, total_items, BATCH_SIZE)):
        batch_urls = urls[i : i + BATCH_SIZE]

        # Качаем картинки
        indices, images = asyncio.run(process_batch(batch_urls, i))

        if images:
            # Превращаем картинки в тензоры
            inputs = processor(images=images, return_tensors="pt").to(device)

            # Генерируем эмбеддинги
            with torch.no_grad():
                output = model.get_image_features(**inputs)
                image_features = output.pooler_output if hasattr(output, 'pooler_output') else output

                # Нормализуем вектора (обязательно для косинусного расстояния)
                image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
                embeddings = image_features.cpu().numpy()

            current_embeddings.append(embeddings)
            current_indices.extend(indices)

        # Сохранение чекпоинта
        if (i + BATCH_SIZE) % SAVE_EVERY == 0 or (i + BATCH_SIZE) >= total_items:
            if current_embeddings:
                batch_emb_array = np.vstack(current_embeddings)
                batch_idx_array = np.array(current_indices)

                end_idx = i + BATCH_SIZE
                np.save(os.path.join(EMB_DIR, f'emb_{end_idx}.npy'), batch_emb_array)
                np.save(os.path.join(EMB_DIR, f'idx_{end_idx}.npy'), batch_idx_array)

                current_embeddings = []
                current_indices = []

# Запускаем!
run_pipeline(goods)

Используем устройство: cuda


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Найдено сохранение. Продолжаем с индекса 160000...


  0%|          | 0/684 [00:00<?, ?it/s]